### Step 1 — Point at the same catalog/schema/volume as Notebook 02

These defaults must match what you used in `02_synthetic_data_generation.ipynb`, or the paths below won't exist.

In [0]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")
dbutils.widgets.text("volume_name", "synthetic_data", "Volume")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

documents_path = f"{volume_path}/documents"
transactions_csv_path = f"{volume_path}/transactions/transactions.csv"

print(f"Documents folder: {documents_path}")
print(f"Transactions CSV: {transactions_csv_path}")

### Step 2 — List the folder and look at raw file metadata

Note this lists exactly one, explicit, known folder -- not the whole Volume, and not the workspace.

In [0]:
files = dbutils.fs.ls(documents_path)
for f in files[:5]:
    print(f"{f.name:45s} {f.size:6d} bytes")
print(f"... {len(files)} files total")

### Step 3 — Read the documents folder as binary

One row per file, raw bytes untouched. This is the representation any file type -- including a PDF -- would take.

In [0]:
binary_df = spark.read.format("binaryFile").load(documents_path)
binary_df.printSchema()
display(binary_df.select("path", "modificationTime", "length"))

The `content` column is raw `bytes`. Decoding it is *your* job -- Spark won't do it for you, because it has no idea whether the bytes are text, a PDF, or an image.

In [0]:
first_file = binary_df.select("path", "content").first()
decoded_text = first_file["content"].decode("utf-8")
print(f"File: {first_file['path']}\n")
print(decoded_text)

### Step 4 — Read the same folder as text

By default, `spark.read.text(...)` gives **one row per line**, not one row per file -- very different from `binaryFile`.

In [0]:
text_lines_df = spark.read.text(documents_path)
print(f"binaryFile row count (one per file): {binary_df.count()}")
print(f"text() row count (one per line):     {text_lines_df.count()}")
display(text_lines_df.limit(8))

Passing `wholetext=True` switches it back to one row per file -- but still typed as a single text `value` column, not raw bytes like `binaryFile`.

In [0]:
text_whole_df = spark.read.text(documents_path, wholetext=True)
print(f"text(wholetext=True) row count: {text_whole_df.count()}")
text_whole_df.printSchema()
text_whole_df.display()

### Step 5 — Read the CSV as structured data

`inferSchema=True` makes Spark scan the data once to guess column types. Compare the inferred types below with the `transactions` Delta table from Notebook 02 -- they may not match exactly, since a CSV file has no native type information.

In [0]:
transactions_csv_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(transactions_csv_path)
)
transactions_csv_df.printSchema()
display(transactions_csv_df.limit(5))

In [0]:
%sql
-- Update the catalog/schema below if you changed the widgets in Step 1
DESCRIBE main.genai_lab.transactions

### Step 6 — Derive metadata the `binaryFile` reader doesn't give you

`binaryFile` knows `path`, `length`, and `modificationTime` -- it has no idea this file's `doc_id` is `1` or its title is "Checking Account Overview". Deriving that requires either parsing the filename (what we do here) or joining back to the `documents` table from Notebook 02 using the filename you generated it with.

In [0]:
from pyspark.sql.functions import element_at, split, regexp_extract

file_manifest_df = (
    binary_df
    .withColumn("file_name", element_at(split("path", "/"), -1))
    .withColumn("doc_id", regexp_extract("file_name", r"^(\d+)_", 1).cast("int"))
    .select("doc_id", "file_name", "length", "modificationTime")
    .orderBy("doc_id")
)
display(file_manifest_df)